<a href="https://colab.research.google.com/github/JoaoVictorDBP/Anime-Recommendation-System/blob/main/AnimeRec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Projeto de Sistema de Recomendação de animes

## API e Requests

In [5]:
!pip install jikanpy-v4 --quiet

In [38]:
import pandas as pd
import numpy as np
from jikanpy import Jikan
import time
import requests
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
jikan = Jikan()

# Objetivos a seguir

1. Coletar Top 1000 Animes
2. Extrair atributos e salvar em DataFrame
3. Fazer pré-processamento (One-Hot + cosine_similarity)
4. Receber lista de animes favoritos do usuário e recomendar


In [8]:
# Analisando as caracteristicas disponíveis de cada anime

anime_details = jikan.anime(1)
print(anime_details['data'].keys())

dict_keys(['mal_id', 'url', 'images', 'trailer', 'approved', 'titles', 'title', 'title_english', 'title_japanese', 'title_synonyms', 'type', 'source', 'episodes', 'status', 'airing', 'aired', 'duration', 'rating', 'score', 'scored_by', 'rank', 'popularity', 'members', 'favorites', 'synopsis', 'background', 'season', 'year', 'broadcast', 'producers', 'licensors', 'studios', 'genres', 'explicit_genres', 'themes', 'demographics'])


In [9]:
# Pega informações dos TOP 1000 animes

anime_list = []

def top_1000():
  for i in range(1, 41):
    print(f"Coletando página {i}...")

    try:
      all_anime = jikan.top('anime', page=i)

      for anime in all_anime['data']:
        anime_info = {
                'mal_id': anime.get('mal_id'),
                'title': anime.get('title'),
                'type': anime.get('type'),
                'score': anime.get('score'),
                'episodes': anime.get('episodes'),
                'rank': anime.get('rank'),
                'popularity': anime.get('popularity'),
                'genres': ', '.join([g['name'] for g in anime.get('genres', [])]),
                'synopsis': anime.get('synopsis'),
                'background': anime.get('background'),
                'themes': ', '.join([t['name'] for t in anime.get('themes', [])]),
                'demographics': ', '.join([d['name'] for d in anime.get('demographics', [])])
        }
        anime_list.append(anime_info)

    except Exception as e:
          print(f"Erro na página {i}: {e}")
          print("Aguardando 10 segundos e tentando novamente...")
          time.sleep(10)
          continue

    time.sleep(1)  # 1 segundo entre requisições

# Executa a função

top_1000()

Coletando página 1...
Coletando página 2...
Coletando página 3...
Coletando página 4...
Coletando página 5...
Coletando página 6...
Coletando página 7...
Coletando página 8...
Coletando página 9...
Coletando página 10...
Coletando página 11...
Coletando página 12...
Coletando página 13...
Coletando página 14...
Coletando página 15...
Coletando página 16...
Coletando página 17...
Coletando página 18...
Coletando página 19...
Coletando página 20...
Coletando página 21...
Coletando página 22...
Coletando página 23...
Coletando página 24...
Coletando página 25...
Coletando página 26...
Coletando página 27...
Coletando página 28...
Coletando página 29...
Coletando página 30...
Coletando página 31...
Coletando página 32...
Coletando página 33...
Coletando página 34...
Coletando página 35...
Coletando página 36...
Coletando página 37...
Coletando página 38...
Coletando página 39...
Coletando página 40...


In [10]:
# Cria DataFrame com os dados obtidos

df_anime = pd.DataFrame(anime_list)
df_anime

,mal_id,title,type,score,episodes,rank,popularity,genres,synopsis,background,themes,demographics
0,52991,Sousou no Frieren,TV,9.29,28.0,1.0,127,"Adventure, Drama, Fantasy",During their decade-long quest to defeat the D...,Sousou no Frieren was released on Blu-ray and ...,,Shounen
1,5114,Fullmetal Alchemist: Brotherhood,TV,9.10,64.0,2.0,3,"Action, Adventure, Drama, Fantasy",After a horrific alchemy experiment goes wrong...,,Military,Shounen
2,9253,Steins;Gate,TV,9.07,24.0,3.0,14,"Drama, Sci-Fi, Suspense",Eccentric scientist Rintarou Okabe has a never...,Steins;Gate is based on 5pb. and Nitroplus' vi...,"Psychological, Time Travel",
3,38524,Shingeki no Kyojin Season 3 Part 2,TV,9.05,10.0,4.0,21,"Action, Drama, Suspense",Seeking to restore humanity's diminishing hope...,Shingeki no Kyojin Season 3 Part 2 adapts cont...,"Gore, Military, Survival",Shounen
4,28977,Gintama°,TV,9.05,51.0,6.0,346,"Action, Comedy, Sci-Fi","Gintoki, Shinpachi, and Kagura return as the f...",,"Gag Humor, Historical, Parody, Samurai",Shounen
...,...,...,...,...,...,...,...,...,...,...,...,...
995,49570,Wu Dong Qian Kun 3rd Season,ONA,7.85,12.0,980.0,9109,"Action, Adventure, Fantasy",Lin Dong continues his journey to find the anc...,,"Historical, Martial Arts",
996,50634,Love Me: Kaede to Suzu The Animation,OVA,7.85,3.0,NaN,5272,Hentai,Kaede is the student council president and is ...,,,
997,1921,Urusei Yatsura 2: Beautiful Dreamer,Movie,7.85,1.0,973.0,4954,"Action, Adventure, Comedy, Drama, Romance, Sci-Fi","Not all is normal in Tomobiki, even by its sta...",,,Shounen
998,283,Akage no Anne,TV,7.85,50.0,983.0,3417,Drama,"Life is not easy for Anne Shirley, an 11-year-...",,Historical,


In [11]:
# Cria Id's para os animes e adiciona no df

df_anime['anime_id'] = range(1, len(df_anime)+1)
df_anime = df_anime[['anime_id'] + [col for col in df_anime.columns if col != 'anime_id']]
df_anime


,anime_id,mal_id,title,type,score,episodes,rank,popularity,genres,synopsis,background,themes,demographics
0,1,52991,Sousou no Frieren,TV,9.29,28.0,1.0,127,"Adventure, Drama, Fantasy",During their decade-long quest to defeat the D...,Sousou no Frieren was released on Blu-ray and ...,,Shounen
1,2,5114,Fullmetal Alchemist: Brotherhood,TV,9.10,64.0,2.0,3,"Action, Adventure, Drama, Fantasy",After a horrific alchemy experiment goes wrong...,,Military,Shounen
2,3,9253,Steins;Gate,TV,9.07,24.0,3.0,14,"Drama, Sci-Fi, Suspense",Eccentric scientist Rintarou Okabe has a never...,Steins;Gate is based on 5pb. and Nitroplus' vi...,"Psychological, Time Travel",
3,4,38524,Shingeki no Kyojin Season 3 Part 2,TV,9.05,10.0,4.0,21,"Action, Drama, Suspense",Seeking to restore humanity's diminishing hope...,Shingeki no Kyojin Season 3 Part 2 adapts cont...,"Gore, Military, Survival",Shounen
4,5,28977,Gintama°,TV,9.05,51.0,6.0,346,"Action, Comedy, Sci-Fi","Gintoki, Shinpachi, and Kagura return as the f...",,"Gag Humor, Historical, Parody, Samurai",Shounen
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,49570,Wu Dong Qian Kun 3rd Season,ONA,7.85,12.0,980.0,9109,"Action, Adventure, Fantasy",Lin Dong continues his journey to find the anc...,,"Historical, Martial Arts",
996,997,50634,Love Me: Kaede to Suzu The Animation,OVA,7.85,3.0,NaN,5272,Hentai,Kaede is the student council president and is ...,,,
997,998,1921,Urusei Yatsura 2: Beautiful Dreamer,Movie,7.85,1.0,973.0,4954,"Action, Adventure, Comedy, Drama, Romance, Sci-Fi","Not all is normal in Tomobiki, even by its sta...",,,Shounen
998,999,283,Akage no Anne,TV,7.85,50.0,983.0,3417,Drama,"Life is not easy for Anne Shirley, an 11-year-...",,Historical,


In [12]:
all_demographics = df_anime['demographics'].str.split(', ', expand=True).stack().unique()
all_demographics

array(['Shounen', '', 'Seinen', 'Shoujo', 'Josei', 'Kids'], dtype=object)

In [13]:
# Função que obtem todos os generos

def get_all_genres(df_anime):
  all_genres = df_anime['genres'].str.split(', ', expand=True).stack().unique()
  return all_genres

all_genres = get_all_genres(df_anime)
all_genres

array(['Adventure', 'Drama', 'Fantasy', 'Action', 'Sci-Fi', 'Suspense',
       'Comedy', 'Supernatural', 'Romance', 'Award Winning', 'Mystery',
       'Sports', 'Slice of Life', '', 'Ecchi', 'Horror', 'Gourmet',
       'Avant Garde', 'Boys Love', 'Hentai', 'Girls Love'], dtype=object)

In [14]:
# Realiza one-hot-coding com os animes e os generos (novo df)

def one_hot_encode(df_anime, all_genres):
  df_encoded = df_anime.copy()
  for genre in all_genres:
    df_encoded[genre] = df_encoded['genres'].apply(lambda x: 1 if genre in x else 0)
  return df_encoded

df_encoded = one_hot_encode(df_anime, all_genres)
df_encoded = df_encoded.drop(columns=['genres','mal_id','type','score','episodes','rank','popularity','synopsis','themes','demographics','background',''])
df_encoded

,anime_id,title,Adventure,Drama,Fantasy,Action,Sci-Fi,Suspense,Comedy,Supernatural,...,Mystery,Sports,Slice of Life,Ecchi,Horror,Gourmet,Avant Garde,Boys Love,Hentai,Girls Love
0,1,Sousou no Frieren,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Fullmetal Alchemist: Brotherhood,1,1,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Steins;Gate,0,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,Shingeki no Kyojin Season 3 Part 2,0,1,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Gintama°,0,0,0,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,Wu Dong Qian Kun 3rd Season,1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,997,Love Me: Kaede to Suzu The Animation,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
997,998,Urusei Yatsura 2: Beautiful Dreamer,1,1,0,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
998,999,Akage no Anne,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [43]:
# Lista de animes favoritos de um certo usuario

fav_list = []

def get_user_fav(user):
  fav_anime = jikan.users(username=user, extension='favorites')
  for anime in fav_anime['data']['anime']:
    anime_title = {
        'title': anime.get('title')
    }
    fav_list.append(anime_title)


username = 'JoaoVictorDBP'
get_user_fav(username)

In [44]:
# Cria df com id's e generos dos anime favoritos

df_fav = pd.DataFrame(fav_list)
df_fav = df_fav.merge(df_anime[['anime_id','title','genres']], on='title', how='left')
df_fav = df_fav.dropna(subset=['anime_id'])
df_fav = df_fav[['anime_id'] + [col for col in df_fav.columns if col != 'anime_id']]

df_fav_encoded = one_hot_encode(df_fav, all_genres)
df_fav_encoded = df_fav_encoded.drop(columns=['genres',''])
df_fav_encoded

,anime_id,title,Adventure,Drama,Fantasy,Action,Sci-Fi,Suspense,Comedy,Supernatural,...,Mystery,Sports,Slice of Life,Ecchi,Horror,Gourmet,Avant Garde,Boys Love,Hentai,Girls Love
0,65,Mob Psycho 100 III,0,0,0,1,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1,Sousou no Frieren,1,1,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,238,Girls Band Cry,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,35,Vinland Saga Season 2,1,1,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,3,Steins;Gate,0,1,0,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
5,405,Tengoku Daimakyou,1,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
6,34,Takopii no Genzai,0,1,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,99,Death Note,0,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
8,19,Koe no Katachi,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,8,Hunter x Hunter (2011),1,0,1,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [45]:
# calcula média do usuario

genre_cols = df_fav_encoded.columns[2:]        # ajuste o índice se necessário
user_mean = df_fav_encoded[genre_cols].mean(axis=0)
print(user_mean)

Adventure        0.4
Drama            0.6
Fantasy          0.2
Action           0.3
Sci-Fi           0.3
Suspense         0.2
Comedy           0.1
Supernatural     0.2
Romance          0.0
Award Winning    0.1
Mystery          0.1
Sports           0.0
Slice of Life    0.0
Ecchi            0.0
Horror           0.0
Gourmet          0.0
Avant Garde      0.0
Boys Love        0.0
Hentai           0.0
Girls Love       0.0
dtype: float64


In [46]:
# comparar similaridade entre média dos favoritos e os outros animes (excluindo os favoritos)

user_vec = user_mean.values.reshape(1, -1)

fav_ids = df_fav_encoded['anime_id'].tolist()
candidates = df_encoded[~df_encoded['anime_id'].isin(fav_ids)].copy()

candidates_genres = candidates[genre_cols].values
scores = cosine_similarity(user_vec, candidates_genres).flatten()

candidates['similaridade'] = scores

top_recs = candidates.sort_values('similaridade', ascending=False).head(50)

print(top_recs[['anime_id', 'title', 'similaridade']])

     anime_id                                              title  similaridade
403       404                             Wu Liuqi: Jiyi Suipian      0.819920
518       519  Interstella5555: The 5tory of The 5ecret 5tar ...      0.814092
583       584                                Mirai Shounen Conan      0.814092
579       580                          Yuusha-Ou GaoGaiGar Final      0.814092
41         42                                       Vinland Saga      0.814092
689       690                              Ginga Nagareboshi Gin      0.814092
973       974      Lupin the IIIrd: Chikemuri no Ishikawa Goemon      0.814092
180       181                                        Banana Fish      0.813489
82         83                                  Tian Guan Cifu Er      0.813489
753       754                                    Wolf's Rain OVA      0.813489
199       200                                     Tian Guan Cifu      0.813489
711       712          Fullmetal Alchemist: Brotherh

# Recomendação de mangás

In [48]:
# Pega informações dos TOP 1000 mangas

manga_list = []

def top_1000_manga():
  for i in range(1, 41):
    print(f"Coletando página {i}...")

    try:
      all_manga = jikan.top('manga', page=i)

      for manga in all_manga['data']:
        manga_info = {
                'mal_id': manga.get('mal_id'),
                'title': manga.get('title'),
                'type': manga.get('type'),
                'score': manga.get('score'),
                'episodes': manga.get('episodes'),
                'rank': manga.get('rank'),
                'popularity': manga.get('popularity'),
                'genres': ', '.join([g['name'] for g in manga.get('genres', [])]),
                'synopsis': manga.get('synopsis'),
                'background': manga.get('background'),
                'themes': ', '.join([t['name'] for t in manga.get('themes', [])]),
                'demographics': ', '.join([d['name'] for d in manga.get('demographics', [])])
        }
        manga_list.append(manga_info)

    except Exception as e:
          print(f"Erro na página {i}: {e}")
          print("Aguardando 10 segundos e tentando novamente...")
          time.sleep(10)
          continue

    time.sleep(1)  # 1 segundo entre requisições

# Executa a função

top_1000_manga()

Coletando página 1...
Coletando página 2...
Coletando página 3...
Coletando página 4...
Coletando página 5...
Coletando página 6...
Coletando página 7...
Coletando página 8...
Coletando página 9...
Coletando página 10...
Coletando página 11...
Coletando página 12...
Coletando página 13...
Coletando página 14...
Coletando página 15...
Coletando página 16...
Coletando página 17...
Coletando página 18...
Coletando página 19...
Coletando página 20...
Coletando página 21...
Coletando página 22...
Coletando página 23...
Coletando página 24...
Coletando página 25...
Coletando página 26...
Coletando página 27...
Coletando página 28...
Coletando página 29...
Coletando página 30...
Coletando página 31...
Coletando página 32...
Coletando página 33...
Coletando página 34...
Coletando página 35...
Coletando página 36...
Coletando página 37...
Coletando página 38...
Coletando página 39...
Coletando página 40...


In [49]:
# Cria DataFrame com os dados obtidos

df_manga = pd.DataFrame(manga_list)
df_manga

,mal_id,title,type,score,episodes,rank,popularity,genres,synopsis,background,themes,demographics
0,2,Berserk,Manga,9.47,None,1.0,1,"Action, Adventure, Award Winning, Drama, Fanta...","Guts, a former mercenary now known as the Blac...",Berserk won the Excellence Award at the sixth ...,"Gore, Military, Psychological",Seinen
1,1706,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,Manga,9.33,None,2.0,22,"Action, Adventure, Mystery, Supernatural","In the American Old West, the world's greatest...",JoJo no Kimyou na Bouken Part 7: Steel Ball Ru...,Historical,"Seinen, Shounen"
2,656,Vagabond,Manga,9.27,None,3.0,12,"Action, Adventure, Award Winning","In 16th-century Japan, Shinmen Takezou is a wi...",Vagabond won the Japan Media Arts Festival Man...,"Historical, Samurai",Seinen
3,13,One Piece,Manga,9.22,None,4.0,4,"Action, Adventure, Fantasy","Gol D. Roger, a man referred to as the King of...",One Piece is the highest-selling manga series ...,,Shounen
4,1,Monster,Manga,9.16,None,5.0,28,"Award Winning, Drama, Mystery","Kenzou Tenma, a renowned Japanese neurosurgeon...",Monster won the Grand Prize at the third Tezuk...,"Adult Cast, Psychological",Seinen
...,...,...,...,...,...,...,...,...,...,...,...,...
995,5655,Kaijuu no Kodomo,Manga,7.94,None,919.0,1158,"Award Winning, Drama, Mystery, Supernatural",Ruka's dad and the other adults who work at th...,Kaijuu no Kodomo was awarded an Excellence Pri...,,Seinen
996,44933,Kanshikan Tsunemori Akane,Manga,7.94,None,927.0,1576,"Action, Drama, Mystery, Sci-Fi","In futuristic Japan, the human soul is analyze...",Kanshikan Tsunemori Akane is adaptation of stu...,"Adult Cast, Detective, Psychological",Shounen
997,3941,Clannad: Tomoyo After,Manga,7.94,None,918.0,2863,"Drama, Romance",Manga adaptation of the visual novel Tomoyo Af...,,,
998,3633,Beast Master,Manga,7.94,None,917.0,293,"Comedy, Romance","Yuiko Kubozuka, a 17-year-old single high scho...",Beast Master was published in English by VIZ M...,,Shoujo


In [50]:
# Cria Id's para os mangas e adiciona no df

df_manga['manga_id'] = range(1, len(df_manga)+1)
df_manga = df_manga[['manga_id'] + [col for col in df_manga.columns if col != 'manga_id']]
df_manga


,manga_id,mal_id,title,type,score,episodes,rank,popularity,genres,synopsis,background,themes,demographics
0,1,2,Berserk,Manga,9.47,None,1.0,1,"Action, Adventure, Award Winning, Drama, Fanta...","Guts, a former mercenary now known as the Blac...",Berserk won the Excellence Award at the sixth ...,"Gore, Military, Psychological",Seinen
1,2,1706,JoJo no Kimyou na Bouken Part 7: Steel Ball Run,Manga,9.33,None,2.0,22,"Action, Adventure, Mystery, Supernatural","In the American Old West, the world's greatest...",JoJo no Kimyou na Bouken Part 7: Steel Ball Ru...,Historical,"Seinen, Shounen"
2,3,656,Vagabond,Manga,9.27,None,3.0,12,"Action, Adventure, Award Winning","In 16th-century Japan, Shinmen Takezou is a wi...",Vagabond won the Japan Media Arts Festival Man...,"Historical, Samurai",Seinen
3,4,13,One Piece,Manga,9.22,None,4.0,4,"Action, Adventure, Fantasy","Gol D. Roger, a man referred to as the King of...",One Piece is the highest-selling manga series ...,,Shounen
4,5,1,Monster,Manga,9.16,None,5.0,28,"Award Winning, Drama, Mystery","Kenzou Tenma, a renowned Japanese neurosurgeon...",Monster won the Grand Prize at the third Tezuk...,"Adult Cast, Psychological",Seinen
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,5655,Kaijuu no Kodomo,Manga,7.94,None,919.0,1158,"Award Winning, Drama, Mystery, Supernatural",Ruka's dad and the other adults who work at th...,Kaijuu no Kodomo was awarded an Excellence Pri...,,Seinen
996,997,44933,Kanshikan Tsunemori Akane,Manga,7.94,None,927.0,1576,"Action, Drama, Mystery, Sci-Fi","In futuristic Japan, the human soul is analyze...",Kanshikan Tsunemori Akane is adaptation of stu...,"Adult Cast, Detective, Psychological",Shounen
997,998,3941,Clannad: Tomoyo After,Manga,7.94,None,918.0,2863,"Drama, Romance",Manga adaptation of the visual novel Tomoyo Af...,,,
998,999,3633,Beast Master,Manga,7.94,None,917.0,293,"Comedy, Romance","Yuiko Kubozuka, a 17-year-old single high scho...",Beast Master was published in English by VIZ M...,,Shoujo


In [ ]:
# continuar...